## 1. Instalación de dependencias

In [10]:
#pip install curl_cffi beautifulsoup4 pandas

## 2. Imports y funciones auxiliares

In [11]:
import re
import os
import time
import pandas as pd
from datetime import date
from bs4 import BeautifulSoup
from curl_cffi import requests as cf_requests

# ──────────────────────────────────────────────
# UTILIDADES
# ──────────────────────────────────────────────

def clean_text(text):
    if not text: return None
    text = text.replace('\xa0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def parse_price(price_text):
    price_text = clean_text(price_text)
    precio = "Consultar"
    expensas = None
    p_match = re.search(r'(USD|ARS|\$)\s?[\$]?\s?([\d\.]+)', price_text)
    if p_match:
        precio = p_match.group(0).strip()
    e_match = re.search(r'[Ee]xp[^$\d]*([\d\.]+)', price_text)
    if not e_match:
        e_match = re.search(r'\+\s?\$?\s?([\d\.]+)', price_text)
    if e_match:
        expensas = e_match.group(0).strip()
    return precio, expensas

def extract_smart_features(row):
    texto = (str(row.get('Descripción', '')) + " " + str(row.get('Detalles', ''))).lower()
    return pd.Series({
        "Amenities":         sum(1 for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"] if x in texto),
        # "Amenities":         1 if any(x in texto for x in ["amenities", "piscina", "pileta", "sum", "parrilla", "gym", "sauna", "laundry"]) else 0,
        "Losa_Central":      1 if any(x in texto for x in ["losa radiante", "calefacción central", "caldera central", "piso radiante"]) else 0,
        "Aire_Acond":        1 if any(x in texto for x in ["aire acondicionado", "split", " a/c", "frío-calor"]) else 0,
        "Apto_Credito":      1 if "apto crédito" in texto or "apto credito" in texto else 0,
        "Cochera":           1 if any(x in texto for x in ["cochera", "espacio guarda coche", "estacionamiento", "guarda coche"]) else 0,
        "Seguridad":         1 if any(x in texto for x in ["vigilancia", "seguridad 24", "tótem", "totem", "encargado"]) else 0,
        "Luminoso":          1 if any(x in texto for x in ["luminoso", "todo luz", "vista abierta", "vista panorámica", "sol"]) else 0,
        "Balcon_Aterrazado": 1 if "aterrazado" in texto or "balcón terraza" in texto else 0,
    })


MESES = r'(enero|febrero|marzo|abril|mayo|junio|julio|agosto|septiembre|octubre|noviembre|diciembre)'

def parse_address(address_raw):
    """
    Maneja todos los casos conocidos de ZonaProp:
    - 'Bolivia al 4400'                          → calle=Bolivia, altura=4400
    - 'Av Luis María Campos 1500'                → calle=Av Luis María Campos, altura=1500
    - 'Junín 1615 piso 13'                       → calle=Junín, altura=1615, piso=13
    - 'SAN JOSE 445. Entre Belgrano y Venezuela' → calle=SAN JOSE, altura=445
    - '11 de Septiembre de 1888 2231'            → calle=11 de Septiembre de 1888, altura=2231
    - 'Torres del Yacht - Juana Manso al 600'    → calle=Juana Manso, altura=600
    - 'Alvear Tower - Azucena Villaflor'         → calle=None, altura=None (sin número)
    - 'El Faro - 3 Ambientes + Dependencia'      → calle=None, altura=None (número no es altura)
    """
    calle = altura = piso = None
    if not address_raw:
        return calle, altura, piso

    try:
        # 1. Limpiar "al " antes de números
        cleaned = re.sub(r'\bal\s+', '', address_raw, flags=re.IGNORECASE)

        # 2. Si tiene guión, quedarse con el fragmento que tenga número de altura válido
        #    Ej: "Alvear Tower - Azucena Villaflor" → ningún fragmento tiene número → None
        #    Ej: "Torres del Yacht - Juana Manso 600 - 2 Ambientes" → "Juana Manso 600"
        #    Ej: "El Faro - 3 Ambientes" → el "3" no es altura válida (< 10), descartar
        if '-' in cleaned:
            fragmentos = [f.strip() for f in cleaned.split('-')]
            candidato = None
            for frag in fragmentos:
                # Buscar fragmento con número >= 100 (alturas reales de CABA son >= 100)
                if re.search(r'\b\d{3,5}\b', frag):
                    candidato = frag
                    break
            if candidato:
                cleaned = candidato
            else:
                # Ningún fragmento tiene altura válida → nombre de edificio sin dirección
                return None, None, None

        # 3. Intentar capturar piso: "Junín 1615 piso 13"
        match_piso = re.search(
            r'([A-Za-záéíóúÁÉÍÓÚñÑü\s\.]+?)\s+(\d{2,5})\s+piso\s+(\d+)',
            cleaned, re.IGNORECASE
        )
        if match_piso:
            num = int(match_piso.group(2))
            if not (1800 <= num <= 1950 and re.search(MESES, cleaned[:match_piso.start(2)], re.IGNORECASE)):
                calle  = match_piso.group(1).strip().strip('-').strip('.').title()
                altura = match_piso.group(2).strip()
                piso   = match_piso.group(3).strip()
                return calle, altura, piso

        # 4. Iterar todos los números y encontrar la altura real
        for m in re.finditer(r'(\d{2,5})(?:[.,\s\-]|$)', cleaned):
            num = int(m.group(1))

            # Descartar números < 100: probablemente ambientes, m², baños, etc.
            if num < 100:
                continue

            # Descartar años históricos si hay un mes en el contexto previo
            contexto_previo = cleaned[:m.start()].lower()
            es_año_historico = (1800 <= num <= 1950) and bool(re.search(MESES, contexto_previo, re.IGNORECASE))
            if es_año_historico:
                continue

            # Este número es la altura real
            calle_raw = cleaned[:m.start()].strip().strip('-').strip('.').strip()
            if not calle_raw:
                continue

            calle  = calle_raw.title()
            altura = str(num)
            # piso queda None
            return calle, altura, piso

    except:
        pass

    return None, None, None

def parse_card(item):
    # Link
    href = item.get('data-to-posting', '')
    if not href:
        a = item.find('a', href=re.compile(r'/propiedades/'))
        href = a['href'] if a else ''
    link = f"https://www.zonaprop.com.ar{href}" if href.startswith('/') else href

    # Precio
    price_tag = item.find(attrs={"data-qa": "POSTING_CARD_PRICE"})
    precio_raw = clean_text(price_tag.text) if price_tag else ""
    precio, _ = parse_price(precio_raw)

    # Expensas
    exp_tag = item.find(attrs={"data-qa": "expensas"})
    expensas = clean_text(exp_tag.text) if exp_tag else None
    expensas = expensas[:-9] if expensas else None

    # ── Dirección: calle+altura y barrio van en tags separados ──
    addr_tag = item.find(class_=re.compile(r'location-address'))
    raw_address = clean_text(addr_tag.text) if addr_tag else ""
    calle, altura, piso = parse_address(raw_address)

    # Barrio/zona (el data-qa="POSTING_CARD_LOCATION")
    barrio_tag = item.find(attrs={"data-qa": "POSTING_CARD_LOCATION"})
    barrio = clean_text(barrio_tag.text) if barrio_tag else None
    barrio = barrio.split(',')[0].strip() if barrio else None

    # Características
    feat_tag = item.find(attrs={"data-qa": "POSTING_CARD_FEATURES"})
    features = clean_text(feat_tag.get_text(separator=' ')) if feat_tag else None

    # Descripción
    desc_tag = item.find(attrs={"data-qa": "POSTING_CARD_DESCRIPTION"})
    desc = clean_text(desc_tag.text) if desc_tag else None

    return link, precio, expensas, calle, altura, piso, barrio, features, desc

print("✅ Funciones utilitarias cargadas.")

✅ Funciones utilitarias cargadas.


## 3. Función principal del scrapper

In [12]:
def run_scrapper_zonaprop(enlace, operacion, max_pages=3):
    BASE = enlace
    OUTPUT_DIR = "output"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    all_data = []
    seen_links = set()

    for page_num in range(1, max_pages + 1):
        url = f"{BASE}.html" if page_num == 1 else f"{BASE}-pagina-{page_num}.html"
        print(f"\n--- PÁGINA {page_num} ---")
        print(f"URL: {url}")

        try:
            r = cf_requests.get(
                url,
                impersonate="chrome120",
                timeout=20,
                headers={
                    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
                    "Accept-Language": "es-AR,es;q=0.9",
                    "Referer": "https://www.zonaprop.com.ar/",
                }
            )
            if r.status_code != 200:
                print(f"❌ Status {r.status_code}. Deteniendo.")
                break
        except Exception as e:
            print(f"❌ Error de red: {e}")
            break

        soup = BeautifulSoup(r.text, 'html.parser')
        items = soup.find_all('div', attrs={"data-qa": "posting PROPERTY"})

        if not items:
            print("⚠️ No se encontraron cards. Fin del listado.")
            break

        print(f"  → {len(items)} propiedades encontradas")

        for item in items:
            try:
                link, precio, expensas, calle, altura, piso, barrio, features, desc = parse_card(item)

                if not link or link in seen_links:
                    continue
                seen_links.add(link)

                print(f"  → {calle} {altura} | {precio}")
                
                all_data.append({
                    "Fecha_Scraping": date.today().isoformat(),
                    "Posting_ID":     item.get('data-id'),
                    "Sito":           'zonaprop',
                    "Operación":      operacion,
                    "Precio":         precio,
                    "Expensas":       expensas,
                    "Calle":          calle,
                    "Altura":         altura,
                    "Piso":           piso,
                    "Barrio":         barrio,       
                    "Detalles":       features,
                    "Descripción":    desc,
                    "Link":           link,
                })

            except Exception as e:
                print(f"    ⚠️ Error en card: {e}")
                continue

        time.sleep(2)

    if not all_data:
        print("⚠️ No se obtuvieron datos.")
        return None

    df = pd.DataFrame(all_data)
    features_df = df.apply(extract_smart_features, axis=1)
    df = pd.concat([df, features_df], axis=1)

    filename = f"zonaprop_{operacion}_{int(time.time())}.tsv"
    filepath = os.path.join(OUTPUT_DIR, filename)
    df.to_csv(filepath, sep='\t', index=False, encoding='utf-8-sig')
    print(f"\n✅ ¡Éxito! Guardado en: {filepath}")
    return df

print("✅ Listo.")

✅ Listo.


## 4. ▶️ Ejecutar el scrapper

# Alquiler

In [13]:
df_alquiler = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal', 'alquiler', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-capital-federal.html


  → 30 propiedades encontradas
  → Av. Directorio 400 | $ 750.000
  → Austria 1900 | USD 1.800
  → Lola Mora 400 | USD 1.300
  → Av. Del Libertador 7000 | USD 6.450
  → Av. Francisco Beiró 3300 | $ 600.000
  → Sarmiento 4300 | $ 990.000
  → Av Del Libertador 800 | $ 1.400.000
  → Crisologo Larralde 2300 | $ 1.400.000
  → Montañeses 2400 | $ 800.000
  → Lola Mora 400 | USD 1.350
  → Vera 1000 | $ 900.000
  → Cervantes 500 | $ 600.000
  → None None | USD 7.100
  → Callao 1300 | USD 2.500
  → Avenida Riestra 5500 | $ 450.000
  → Avenida Scalabrini Ortiz 1600 | USD 750
  → None None | USD 900
  → Montañeses 2477 | USD 600
  → None None | USD 1.300
  → None None | USD 800
  → Forest 400 | $ 680.000
  → Republica Arabe Siria 3000 | USD 1.600
  → Maure 1600 | USD 2.000
  → Sinclair 3026 | $ 580.000
  → Av Del Libertador 8500 | $ 1.000.000
  → Arevalo 1300 | USD 900
  → Azucena Villaflor 500 | USD 10.000
  → Azucena Villaflor 600 | USD 7.000
  → None None | USD 1.500
  → Suipacha 1200 | USD 1.

In [14]:
if df_alquiler is not None:
    print(f"Total propiedades: {len(df_alquiler)}")
    display(df_alquiler.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 90


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-10,58497662,zonaprop,alquiler,$ 750.000,$ 258.000,Av. Directorio,400,None,Caballito,...,Alquiler - 2 ambientes c/ balcón a mts del sub...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,0,0
1,2026-04-10,57722860,zonaprop,alquiler,USD 1.800,$ 295.000,Austria,1900,None,Recoleta,...,Departamento de 2 ambientes con balcón en la e...,https://www.zonaprop.com.ar/propiedades/clasif...,4,0,1,0,1,1,1,0
2,2026-04-10,58719464,zonaprop,alquiler,USD 1.300,$ 289.000,Lola Mora,400,None,Puerto Madero,...,Exclusivo monoambiente en Zencity — Torre Ónix...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,1,1,1,0
3,2026-04-10,57846416,zonaprop,alquiler,USD 6.450,$ 1.900.000,Av. Del Libertador,7000,None,Núñez,...,Alquiler de departamento 4 ambientes con depen...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,0,0,1,1,1,0
4,2026-04-10,58768426,zonaprop,alquiler,$ 600.000,$ 105.000,Av. Francisco Beiró,3300,None,Villa Devoto,...,Alquiler - solo contacto por whatsapp. Alquile...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
5,2026-04-10,58681413,zonaprop,alquiler,$ 990.000,$ 215.000,Sarmiento,4300,None,Almagro Norte,...,Excelente departamento 3 ambientes en alquiler...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
6,2026-04-10,58691385,zonaprop,alquiler,$ 1.400.000,$ 500.000,Av Del Libertador,800,None,Retiro,...,"Ubicado en el prestigioso barrio de Retiro, en...",https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,1,1,0
7,2026-04-10,58775490,zonaprop,alquiler,$ 1.400.000,$ 250.000,Crisologo Larralde,2300,None,Núñez,...,Broker: Lali +54 9 11 Ver datosAmplio 2 ambien...,https://www.zonaprop.com.ar/propiedades/clasif...,2,0,0,0,0,0,1,0
8,2026-04-10,58511505,zonaprop,alquiler,$ 800.000,$ 100.000,Montañeses,2400,None,Belgrano,...,Monoambiente al frente con salida a balcón. Co...,https://www.zonaprop.com.ar/propiedades/clasif...,2,0,0,0,0,0,0,0
9,2026-04-10,58125227,zonaprop,alquiler,USD 1.350,$ 275.000,Lola Mora,400,None,Puerto Madero,...,Alquiler Departamento Monoambiente en Harbour ...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,1,0,1,0



📊 % con cada característica:
Losa_Central         13.3%
Aire_Acond           34.4%
Apto_Credito          0.0%
Cochera              45.6%
Seguridad            40.0%
Luminoso             74.4%
Balcon_Aterrazado     4.4%
dtype: object


# Alquiler Temporal

In [15]:
df_alquiler_temporal = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal', 'alquiler_temporal', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-federal.html


  → 27 propiedades encontradas
  → Adolfo Alsina 1928 | USD 400
  → O¨Higgins 4600 | USD 800
  → Rodriguez Peña 1800 | USD 2.000
  → Guemes 3600 | USD 1.500
  → Emilio Ravignani 2300 | USD 1.100
  → Guemes 3000 | USD 1.500
  → Lola Mora 400 | USD 3.800
  → Lafinur 3200 | USD 1.900
  → Arenales 2300 | USD 1.150
  → Soldado De La Independencia 900 | USD 600
  → Sánchez De Bustamante 2100 | USD 1.500
  → Jorge Newberry 2500 | USD 1.200
  → Mansilla 2600 | USD 850
  → Av Rivadavia 5800 | USD 650
  → Santa Fe 5381 | USD 1.100
  → Av Luis Maria Campos 300 | USD 750
  → Miller 3100 | USD 600
  → Thompson 733 | USD 1.250
  → Av Santa Fe 5000 | USD 800
  → Av Libertador 4800 | USD 850
  → Amenabar 3100 | USD 1.200
  → Zelarrayan 1600 | USD 600
  → Av. Juana Manso 1500 | USD 2.200
  → Vera 800 | USD 700
  → Salguero 3000 | $ 920.000
  → Avenida San Juan 2400 | $ 650.000
  → Teodoro Garcia 2800 | USD 1.100

--- PÁGINA 2 ---
URL: https://www.zonaprop.com.ar/departamentos-alquiler-temporal-capital-

In [16]:
if df_alquiler_temporal is not None:
    print(f"Total propiedades: {len(df_alquiler_temporal)}")
    display(df_alquiler_temporal.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_alquiler_temporal[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 87


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-10,52533377,zonaprop,alquiler_temporal,USD 400,$ 100.000,Adolfo Alsina,1928,None,Congreso,...,Alquiler temporario de Departamento 1 ambiente...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,1,0,0,0,1,0
1,2026-04-10,58787934,zonaprop,alquiler_temporal,USD 800,None,O¨Higgins,4600,None,Núñez,...,-La unidad se encuentra en el piso 5 del edifi...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,1,0,1,0
2,2026-04-10,57933318,zonaprop,alquiler_temporal,USD 2.000,$ 182.000,Rodriguez Peña,1800,None,Recoleta,...,Departamento 3 Ambientes + Dependencia | Amobl...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
3,2026-04-10,57170383,zonaprop,alquiler_temporal,USD 1.500,None,Guemes,3600,None,Palermo,...,Departamento de un ambiente para alquiler temp...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,0,1
4,2026-04-10,53421305,zonaprop,alquiler_temporal,USD 1.100,$ 200.000,Emilio Ravignani,2300,None,Palermo Hollywood,...,Alquiler temporario de Departamento 3 ambiente...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,0,0,0,1
5,2026-04-10,58576225,zonaprop,alquiler_temporal,USD 1.500,None,Guemes,3000,None,Recoleta,...,Excelente departamento de tres ambientes con s...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,0,0
6,2026-04-10,58810877,zonaprop,alquiler_temporal,USD 3.800,None,Lola Mora,400,None,Puerto Madero,...,Alquiler temporario de 3 ambientes amueblado e...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,0,0,1,1,0,0
7,2026-04-10,58206763,zonaprop,alquiler_temporal,USD 1.900,None,Lafinur,3200,None,Palermo Chico,...,Alquiler temporarioperiodopreciomayousd1. 900J...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,1,0,0,0,1,0
8,2026-04-10,57872253,zonaprop,alquiler_temporal,USD 1.150,None,Arenales,2300,None,Recoleta,...,Departamento de dos ambientes para alquiler te...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,1,0,1,1,1,0
9,2026-04-10,58604004,zonaprop,alquiler_temporal,USD 600,$ 110.000,Soldado De La Independencia,900,None,Las Cañitas,...,Alquiler temporarioperiodopreciofebrerousd600P...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,0,1,0



📊 % con cada característica:
Losa_Central          8.0%
Aire_Acond           60.9%
Apto_Credito          0.0%
Cochera              25.3%
Seguridad            19.5%
Luminoso             63.2%
Balcon_Aterrazado    12.6%
dtype: object


# Venta

In [17]:
df_venta = run_scrapper_zonaprop('https://www.zonaprop.com.ar/departamentos-venta-capital-federal', 'venta', max_pages=3)


--- PÁGINA 1 ---
URL: https://www.zonaprop.com.ar/departamentos-venta-capital-federal.html


  → 25 propiedades encontradas
  → Nicaragua 6000 | USD 122.000
  → Santa Fé 5300 | USD 145.000
  → John F. Kennedy 2900 | USD 890.000
  → Viamonte 1866 | USD 95.000
  → Ortega Y Gasset 1500 | USD 167.000
  → Balcarce 300 | USD 82.500
  → Billinghurst 2400 | USD 415.000
  → Ortega Y Gasset 1900 | USD 467.400
  → Sanabria 1900 | USD 155.000
  → Quesada 2900 | USD 140.000
  → Jean Jaures 800 | USD 198.900
  → Jose Marti 600 | USD 135.000
  → Angel Gallardo 700 | USD 189.000
  → Cuba 2300 | USD 395.000
  → Fernández De Enciso 4200 | USD 165.000
  → Avenida Triunvirato 3176 | USD 106.000
  → Austria 2500 | USD 98.000
  → Deheza 2300 | USD 192.000
  → Hipólito Yrigoyen 1900 | USD 74.000
  → Beruti 3657 | USD 550.000
  → Arenales 1300 | USD 598.000
  → Araoz 2500 | USD 308.000
  → Mendoza 2900 | USD 176.000
  → Bonifacio Jose 1900 | USD 330.000
  → Cucha Cucha 1161 | USD 145.000

--- PÁGINA 2 ---
URL: https://www.zonaprop.com.ar/departamentos-venta-capital-federal-pagina-2.html
  → 25 propie

In [18]:
if df_venta is not None:
    print(f"Total propiedades: {len(df_venta)}")
    display(df_venta.head(10))

    cols = ["Losa_Central","Aire_Acond","Apto_Credito",
            "Cochera","Seguridad","Luminoso","Balcon_Aterrazado"]
    print("\n📊 % con cada característica:")
    print((df_venta[cols].mean() * 100).round(1).astype(str) + "%")

Total propiedades: 75


,Fecha_Scraping,Posting_ID,Sito,Operación,Precio,Expensas,Calle,Altura,Piso,Barrio,...,Descripción,Link,Amenities,Losa_Central,Aire_Acond,Apto_Credito,Cochera,Seguridad,Luminoso,Balcon_Aterrazado
0,2026-04-10,58259698,zonaprop,venta,USD 122.000,$ 160.000,Nicaragua,6000,None,Palermo Hollywood,...,"En una cuadra arbolada de Palermo Hollywood, e...",https://www.zonaprop.com.ar/propiedades/clasif...,5,0,0,0,1,0,1,0
1,2026-04-10,58790439,zonaprop,venta,USD 145.000,$ 150.000,Santa Fé,5300,None,Palermo,...,Amplio departamento 3 ambientes | frente con b...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
2,2026-04-10,58729087,zonaprop,venta,USD 890.000,$ 1.650.000,John F. Kennedy,2900,None,Palermo Nuevo,...,"Piso de 233 m2 más balcones, ubicado a metros ...",https://www.zonaprop.com.ar/propiedades/clasif...,1,1,1,0,1,1,1,1
3,2026-04-10,58655266,zonaprop,venta,USD 95.000,$ 162.600,Viamonte,1866,None,Balvanera,...,Venta de Departamento 2 ambientes en Balvanera...,https://www.zonaprop.com.ar/propiedades/clasif...,0,0,0,0,0,0,1,0
4,2026-04-10,58687703,zonaprop,venta,USD 167.000,None,Ortega Y Gasset,1500,None,Las Cañitas,...,Monoambiente divisible al contrafrente con sal...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,0,0,0,0,0,0
5,2026-04-10,56290368,zonaprop,venta,USD 82.500,$ 159.000,Balcarce,300,None,San Telmo,...,Departamento muy luminoso en edificio moderno ...,https://www.zonaprop.com.ar/propiedades/clasif...,1,0,0,0,0,0,1,0
6,2026-04-10,58603614,zonaprop,venta,USD 415.000,$ 820.000,Billinghurst,2400,None,Palermo,...,Hermoso departamento de 4 ambientes con depend...,https://www.zonaprop.com.ar/propiedades/clasif...,0,1,1,0,1,0,1,0
7,2026-04-10,58262928,zonaprop,venta,USD 467.400,None,Ortega Y Gasset,1900,None,Las Cañitas,...,4amb + dependencia+ patiofecha de entrega: ago...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,0,0,1,0,1,0
8,2026-04-10,58754076,zonaprop,venta,USD 155.000,None,Sanabria,1900,None,Monte Castro,...,Sanabria 1956 – Gran 2 ambientes de categoría ...,https://www.zonaprop.com.ar/propiedades/clasif...,3,0,1,0,1,0,1,0
9,2026-04-10,58683961,zonaprop,venta,USD 140.000,$ 280.000,Quesada,2900,None,Núñez,...,Corredor Responsable: power bienes raíces S. R...,https://www.zonaprop.com.ar/propiedades/clasif...,5,0,1,1,1,0,0,0



📊 % con cada característica:
Losa_Central         22.7%
Aire_Acond           40.0%
Apto_Credito         16.0%
Cochera              45.3%
Seguridad            18.7%
Luminoso             78.7%
Balcon_Aterrazado    10.7%
dtype: object
